#### Model Training and PEFT Fine-Tuning
This section uses a weak baseline first, then a pure NumPy LoRA-style PEFT demo so the notebook stays runnable without extra deep learning packages.

In [2]:
import numpy as np
import pandas as pd

In [3]:
feature_cols = [f'X{i}' for i in range(1, 17)]
target_col = 'y'
df = pd.read_csv('BEED_Data.csv', usecols=feature_cols + [target_col])
feat_Df = df[feature_cols]

In [6]:
rng = np.random.default_rng(42)
X = feat_Df.to_numpy(dtype=np.float32)
y = df[target_col].to_numpy(dtype=np.int64)

cls = np.unique(y)
trainIdx = []
tstIdx = []
valIdx = []

for c in cls:
    cIdx = np.where(y ==  c)[0]
    cIdx = rng.permutation(cIdx)
    n_smps = len(cIdx)
    n_trn = int(0.7 * n_smps)
    n_val = int(0.15 * n_smps)
    trainIdx.extend(cIdx[:n_trn])
    valIdx.extend(cIdx[n_trn:n_trn+n_val])
    tstIdx.extend(cIdx[n_trn+n_val:])

trainIdx = rng.permutation(trainIdx)
valIdx = rng.permutation(valIdx)
tstIdx = rng.permutation(tstIdx)

X_trnRaw , y_trn = X[trainIdx], y[trainIdx]
X_valRaw, y_val =  X[valIdx], y[valIdx]
X_testRaw, y_tst = X[tstIdx], y[tstIdx]

trnMean = X_trnRaw.mean(axis=0)
trnStd = X_trnRaw.std(axis=0) + 1e-6
X_trn = (X_trnRaw - trnMean) / trnStd
X_val = (X_valRaw - trnMean) / trnStd
X_tst = (X_testRaw - trnMean) / trnStd

X_trn_weak = X_trn[:, :8]
X_val_weak = X_val[:, :8]
X_tst_weak = X_tst[:, :8]
nCls = len(cls)
inpDim = X_trn.shape[1]
wkDim = X_trn_weak.shape[1]

In [9]:
def oneHot(lbls, nmCls):
    encd = np.zeros((len(lbls), nmCls), dtype=np.float32)
    encd[np.arange(len(lbls)), lbls] = 1.0
    return encd

In [10]:
def sftMax(log):
    log = log - log.max(axis=1, keepdims=True)
    expLogs = np.exp(log)
    return expLogs / expLogs.sum(axis=1, keepdims=True)

In [11]:
def acc_Logs(lgs, lbls):
    return float((lgs.argmax(axis=1) == lbls).mean())

In [13]:
print(f"Train Set Size : {len(X_trn)}")
print(f"Value Set Size : {len(X_val)}")
print(f"Test Set Size : {len(X_tst)}")
print(f"Feature dimensions: full={inpDim}, weak_baseline={wkDim}")
print(f"Classes: {cls.tolist()}")

Train Set Size : 5600
Value Set Size : 1200
Test Set Size : 1200
Feature dimensions: full=16, weak_baseline=8
Classes: [0, 1, 2, 3]


In [18]:
dumb = np.random.default_rng(7)
W_dumb = dumb.normal(scale=0.05, size=(nCls, wkDim)).astype(np.float32)
b_dumb = np.zeros(nCls, dtype=np.float32)
dumb_lr = 0.08
dumbEpcs = 12
btchSz = 128
imprv = {"val_acc": -1.0, "W": None, "b": None}

for ep in range(dumbEpcs):
    order = dumb.permutation(len(X_trn_weak))
    X_shuf = X_trn_weak[order]
    y_shuf = y_trn[order]
    for st in range(0, len(X_shuf), btchSz):
        xb = X_shuf[st:st + btchSz]
        yb = y_shuf[st:st + btchSz]
        lgts = xb @ W_dumb.T + b_dumb
        probs = sftMax(lgts)
        y_one_hot = oneHot(yb, nCls)
        grad_lgts = (probs - y_one_hot) / len(xb)
        grad_W = grad_lgts.T @ xb + 0.005 * W_dumb
        grad_b = grad_lgts.sum(axis=0)
        W_dumb -= dumb_lr * grad_W
        b_dumb -= dumb_lr * grad_b

    val_lgts = X_val_weak @ W_dumb.T + b_dumb
    val_acc = acc_Logs(val_lgts, y_val)
    if val_acc > imprv["val_acc"]:
        imprv["val_acc"] = val_acc
        imprv["W"] = W_dumb.copy()
        imprv["b"] = b_dumb.copy()
    if ep in {0, 3, 7, 11}:
        train_acc = acc_Logs(X_trn_weak @ W_dumb.T + b_dumb, y_trn)
        print(f"Epoch {ep + 1:02d} | Train Accuracy = {train_acc:.3f} | Value Accuracy = {val_acc:.3f}")

W_dumb = imprv["W"]
b_dumb = imprv["b"]
dumbtstAcc = acc_Logs(X_tst_weak @ W_dumb.T + b_dumb, y_tst)
print(f"Dumb test accuracy: {dumbtstAcc:.3f}")

Epoch 01 | Train Accuracy = 0.424 | Value Accuracy = 0.407
Epoch 04 | Train Accuracy = 0.447 | Value Accuracy = 0.440
Epoch 08 | Train Accuracy = 0.455 | Value Accuracy = 0.455
Epoch 12 | Train Accuracy = 0.453 | Value Accuracy = 0.445
Dumb test accuracy: 0.463
